# **Encoding and Decoding using Attention Mechanism**

In [ ]:
import numpy as np

from tensorflow.keras.models import Model

from tensorflow.keras.layers import (
    Input,
    LSTM,
    Embedding,
    Dense,
    Attention,
    Concatenate
)

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# ***Prepare Dataset***

In [ ]:

english_sentences = [
    "i love ai",
    "i love deep learning",
    "how are you",
    "good morning"
]

french_sentences = [
    "start j aime ai end",
    "start j aime apprentissage profond end",
    "start comment allez vous end",
    "start bonjour end"
]

print("English Sentences:")
print(english_sentences)

print("French Sentences:")
print(french_sentences)

English Sentences:
['i love ai', 'i love deep learning', 'how are you', 'good morning']
French Sentences:
['start j aime ai end', 'start j aime apprentissage profond end', 'start comment allez vous end', 'start bonjour end']


# ***Tokenization***

In [ ]:
eng_tokenizer = Tokenizer()
eng_tokenizer.fit_on_texts(english_sentences)

fra_tokenizer = Tokenizer()
fra_tokenizer.fit_on_texts(french_sentences)

print("English Vocabulary:")
print(eng_tokenizer.word_index)

print("French Vocabulary:")
print(fra_tokenizer.word_index)

English Vocabulary:
{'i': 1, 'love': 2, 'ai': 3, 'deep': 4, 'learning': 5, 'how': 6, 'are': 7, 'you': 8, 'good': 9, 'morning': 10}
French Vocabulary:
{'start': 1, 'end': 2, 'j': 3, 'aime': 4, 'ai': 5, 'apprentissage': 6, 'profond': 7, 'comment': 8, 'allez': 9, 'vous': 10, 'bonjour': 11}


# ***Convert Sentences to Sequences***

In [ ]:
encoder_input = eng_tokenizer.texts_to_sequences(
    english_sentences
)

decoder_input = fra_tokenizer.texts_to_sequences(
    french_sentences
)

print("English Sequences:")
print(encoder_input)

print("French Sequences:")
print(decoder_input)

English Sequences:
[[1, 2, 3], [1, 2, 4, 5], [6, 7, 8], [9, 10]]
French Sequences:
[[1, 3, 4, 5, 2], [1, 3, 4, 6, 7, 2], [1, 8, 9, 10, 2], [1, 11, 2]]


# ***Padding***

In [ ]:
encoder_input = pad_sequences(
    encoder_input,
    padding='post'
)

decoder_input = pad_sequences(
    decoder_input,
    padding='post'
)

print("Padded English Sequences:")
print(encoder_input)

print("Padded French Sequences:")
print(decoder_input)

Padded English Sequences:
[[ 1  2  3  0]
 [ 1  2  4  5]
 [ 6  7  8  0]
 [ 9 10  0  0]]
Padded French Sequences:
[[ 1  3  4  5  2  0]
 [ 1  3  4  6  7  2]
 [ 1  8  9 10  2  0]
 [ 1 11  2  0  0  0]]


# ***Build Encoder***

In [ ]:
encoder_inputs = Input(shape=(None,))

encoder_embedding = Embedding(
    input_dim=len(eng_tokenizer.word_index)+1,
    output_dim=64
)(encoder_inputs)

encoder_outputs, state_h, state_c = LSTM(
    units=64,
    return_sequences=True,
    return_state=True
)(encoder_embedding)

print("Encoder created successfully")

Encoder created successfully


# ***Build Decoder***

In [ ]:
decoder_inputs = Input(shape=(None,))

decoder_embedding = Embedding(
    input_dim=len(fra_tokenizer.word_index)+1,
    output_dim=64
)(decoder_inputs)

decoder_outputs, _, _ = LSTM(
    units=64,
    return_sequences=True,
    return_state=True
)(
    decoder_embedding,
    initial_state=[state_h, state_c]
)

print("Decoder created successfully")

Decoder created successfully


# ***Attention Layer***

In [ ]:
attention = Attention()

attention_output = attention(
    [decoder_outputs, encoder_outputs]
)

print("Attention layer created successfully")

Attention layer created successfully


# ***Combine Decoder Output and Attention Output***

In [ ]:
decoder_combined = Concatenate(axis=-1)(
    [decoder_outputs, attention_output]
)

print("Decoder combined successfully")

Decoder combined successfully


# ***Final Prediction Layer***

In [ ]:
output = Dense(
    len(fra_tokenizer.word_index)+1,
    activation='softmax'
)(decoder_combined)

print("Prediction layer created successfully")

Prediction layer created successfully


# ***Build Model***

In [ ]:
model = Model(
    [encoder_inputs, decoder_inputs],
    output
)

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ input_layer_3       │ (None, None)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_2         │ (None, None, 64)  │        704 │ input_layer_2[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding_3         │ (None, None, 64)  │        768 │ input_layer_3[0]… │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_2 (LSTM)       │ [(None, None,     │     33,024 │ embedding_2[0][0] │
│                     │ 64), (None, 64),  │            │                   │
│                     │ (None, 64)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_3 (LSTM)       │ [(None, None,     │     33,024 │ embedding_3[0][0… │
│                     │ 64), (None, 64),  │            │ lstm_2[0][1],     │
│                     │ (None, 64)]       │            │ lstm_2[0][2]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ attention           │ (None, None, 64)  │          0 │ lstm_3[0][0],     │
│ (Attention)         │                   │            │ lstm_2[0][0]      │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, None, 128) │          0 │ lstm_3[0][0],     │
│ (Concatenate)       │                   │            │ attention[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 12)  │      1,548 │ concatenate[0][0] │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 69,068 (269.80 KB)

 Trainable params: 69,068 (269.80 KB)

 Non-trainable params: 0 (0.00 B)

# ***Model Compilation***

In [ ]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print("Model compiled successfully")

Model compiled successfully


# ***Train the Model***

In [ ]:
decoder_target = np.expand_dims(decoder_input, -1)

history = model.fit(
    [encoder_input, decoder_input],
    decoder_target,
    epochs=100,
    batch_size=2,
    verbose=1
)

print("Model trained successfully")

Epoch 1/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 4s 28ms/step - accuracy: 0.0833 - loss: 2.4844
Epoch 2/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.2917 - loss: 2.4703
Epoch 3/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.3333 - loss: 2.4596
Epoch 4/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3750 - loss: 2.4463
Epoch 5/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.3750 - loss: 2.4298
Epoch 6/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - accuracy: 0.3750 - loss: 2.4138
Epoch 7/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.3750 - loss: 2.3985
Epoch 8/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - accuracy: 0.3750 - loss: 2.3730
Epoch 9/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 40ms/step - accuracy: 0.3750 - loss: 2.3512
Epoch 10/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.3750 - loss: 2.3206
Epoch 11/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.3750 - loss: 2.2830
Epoch 12/100
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.3750 - lo

# ***Test the Model***

In [ ]:
prediction = model.predict(
    [encoder_input, decoder_input]
)

print("Prediction generated successfully")
print(prediction.shape)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 311ms/step
Prediction generated successfully
(4, 6, 12)


# ***Convert Predictions to Words***

In [ ]:
predicted_ids = np.argmax(prediction, axis=-1)

print("Predicted Token IDs:")
print(predicted_ids)

Predicted Token IDs:
[[ 1  3  4  5  2  0]
 [ 1  3  4  6  7  2]
 [ 1  8  9 10  2  0]
 [ 1 11  2  0  0  0]]


# ***Display Predicted Sentences***

In [ ]:
reverse_fra_vocab = {
    v: k for k, v in fra_tokenizer.word_index.items()
}

for seq in predicted_ids:
    sentence = []

    for token in seq:
        word = reverse_fra_vocab.get(token, "")
        sentence.append(word)

    print(" ".join(sentence))

start j aime ai end 
start j aime apprentissage profond end
start comment allez vous end 
start bonjour end   
